# Fine-grained Tool Calling

When handling requests from Claude, in addition to the `ContentBlockDelta`, there is also an `InputJsonEvent` that could be returned 
Contains an:
- `partial_json`
- `snapshot`

In [ ]:
from utils.util_funcs import * 

client, model = api_client_setup()

In [ ]:
def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


In [ ]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [ ]:
save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)

In [ ]:
def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)

In [6]:
messages = []

add_user_message( 
    messages, 
    'Create and save a fake computer science article'
)

run_conversation(messages, tools = [
    save_article_schema
])


>>> Tool Call: "save_article"
{"abstract": "A novel quantum-inspired algorithm for optimizing neural network training through distributed consciousness simulation.", "meta": {"word_count":4500,"review":"This paper presents an innovative approach to neural network optimization by leveraging quantum-inspired principles combined with distributed computing architectures. The authors introduce a fascinating concept of consciousness simulation that purportedly accelerates convergence rates by 340% compared to traditional backpropagation methods. While the theoretical framework is compelling and the experimental results appear promising, the paper lacks sufficient mathematical rigor in several key proofs and the reproducibility concerns are noteworthy given the proprietary nature of their implementation. The writing is generally clear though somewhat speculative in places, particularly in the consciousness simulation sections where the connection to actual quantum mechanics remains tenuous. 

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [BetaToolUseBlock(id='toolu_01H1HmUhCPUiQXjhgvFkpguE', input={'abstract': 'A novel quantum-inspired algorithm for optimizing neural network training through distributed consciousness simulation.', 'meta': {'word_count': 4500, 'review': 'This paper presents an innovative approach to neural network optimization by leveraging quantum-inspired principles combined with distributed computing architectures. The authors introduce a fascinating concept of consciousness simulation that purportedly accelerates convergence rates by 340% compared to traditional backpropagation methods. While the theoretical framework is compelling and the experimental results appear promising, the paper lacks sufficient mathematical rigor in several key proofs and the reproducibility concerns are noteworthy given the proprietary nature of their implementation. The wr